# AI Resume Analyzer & Job Recommendation System

### Technologies Used

- Python
- Streamlit (Deployment)
- pypdf
- python-docx
- spaCy
- TF-IDF
- Sentence Transformers
- SQLite
- Gemini API

In [4]:
# Install required libraries
!pip install -q pypdf python-docx spacy pandas numpy scikit-learn

# Download the English spaCy model
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
import os
import re
import sqlite3

import pandas as pd
import numpy as np

import nltk
import spacy

from google.colab import files
from pypdf import PdfReader
from docx import Document

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [8]:
nltk.download("punkt")
nltk.download("stopwords")

!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
DATABASE_NAME = "resume_analyzer.db"

UPLOAD_FOLDER = "uploads"

SUPPORTED_FILES = [".pdf", ".docx"]

ATS_THRESHOLD = 75



In [10]:
import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

In [11]:
folders = [
    "uploads",
    "reports",
    "datasets"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders Created Successfully")

Folders Created Successfully


In [12]:
conn = sqlite3.connect(DATABASE_NAME)

cursor = conn.cursor()

print("Database Created Successfully")

Database Created Successfully


In [13]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS resumes (

    resume_id INTEGER PRIMARY KEY AUTOINCREMENT,

    file_name TEXT,

    candidate_name TEXT,

    email TEXT,

    phone TEXT,

    ats_score REAL,

    recommended_job TEXT,

    upload_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP

)
""")

In [14]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS skills (

    skill_id INTEGER PRIMARY KEY AUTOINCREMENT,

    resume_id INTEGER,

    skill_name TEXT,

    FOREIGN KEY(resume_id)
    REFERENCES resumes(resume_id)

)
""")

In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS jobs (

    job_id INTEGER PRIMARY KEY AUTOINCREMENT,

    job_title TEXT,

    required_skills TEXT,

    experience_level TEXT

)
""")

In [16]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS courses (

    course_id INTEGER PRIMARY KEY AUTOINCREMENT,

    course_name TEXT,

    skill TEXT,

    platform TEXT,

    course_link TEXT

)
""")

In [17]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS ats_keywords (

    keyword_id INTEGER PRIMARY KEY AUTOINCREMENT,

    keyword TEXT,

    category TEXT

)
""")

In [18]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS career_paths (

    path_id INTEGER PRIMARY KEY AUTOINCREMENT,

    current_role TEXT,

    next_role TEXT,

    required_skills TEXT,

    recommended_courses TEXT,

    estimated_time TEXT,

    difficulty TEXT,

    average_salary TEXT,

    industry TEXT

)
""")

In [19]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS certifications (

    certification_id INTEGER PRIMARY KEY AUTOINCREMENT,

    certification_name TEXT,

    provider TEXT,

    primary_skill TEXT,

    difficulty TEXT,

    target_role TEXT,

    estimated_duration TEXT,

    certificate_valid TEXT,

    official_url TEXT

)
""")

In [20]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS interview_questions (

    question_id INTEGER PRIMARY KEY AUTOINCREMENT,

    job_role TEXT,

    skill TEXT,

    difficulty TEXT,

    question_type TEXT,

    question TEXT,

    expected_key_points TEXT,

    sample_answer TEXT

)
""")

In [21]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS salary_data (

    salary_id INTEGER PRIMARY KEY AUTOINCREMENT,

    job_role TEXT,

    location TEXT,

    company_type TEXT,

    experience TEXT,

    minimum_salary_lpa REAL,

    maximum_salary_lpa REAL,

    average_salary_lpa REAL,

    required_skills TEXT,

    education TEXT,

    employment_type TEXT

)
""")

In [22]:
conn.commit()

print("All Database Tables Created Successfully")

All Database Tables Created Successfully


In [23]:
cursor.execute(
"""
SELECT name
FROM sqlite_master
WHERE type='table';
"""
)

tables = cursor.fetchall()

print(tables)

[('resumes',), ('sqlite_sequence',), ('skills',), ('jobs',), ('courses',), ('ats_keywords',), ('career_paths',), ('certifications',), ('interview_questions',), ('salary_data',)]


In [24]:
from google.colab import files

uploaded = files.upload()

Saving salary_data.csv to salary_data.csv
Saving interview_questions.csv to interview_questions.csv
Saving certifications.csv to certifications.csv
Saving career_paths.csv to career_paths.csv
Saving courses.csv to courses.csv
Saving ats_keywords.csv to ats_keywords.csv
Saving skills.csv to skills.csv
Saving jobs.csv to jobs.csv


In [25]:
import pandas as pd

jobs_df = pd.read_csv("jobs.csv")
skills_df = pd.read_csv("skills.csv")
courses_df = pd.read_csv("courses.csv")
ats_df = pd.read_csv("ats_keywords.csv")
career_df = pd.read_csv("career_paths.csv")
cert_df = pd.read_csv("certifications.csv")
interview_df = pd.read_csv("interview_questions.csv")
salary_df = pd.read_csv("salary_data.csv")

In [26]:
print("Jobs:", jobs_df.shape)
print("Skills:", skills_df.shape)
print("Courses:", courses_df.shape)
print("ATS:", ats_df.shape)
print("Career Paths:", career_df.shape)
print("Certifications:", cert_df.shape)
print("Interview Questions:", interview_df.shape)
print("Salary:", salary_df.shape)

Jobs: (300, 11)
Skills: (498, 8)
Courses: (348, 9)
ATS: (220, 8)
Career Paths: (200, 9)
Certifications: (200, 9)
Interview Questions: (500, 8)
Salary: (600, 11)


In [27]:
jobs_df.to_sql("jobs", conn, if_exists="replace", index=False)

skills_df.to_sql("skills", conn, if_exists="replace", index=False)

courses_df.to_sql("courses", conn, if_exists="replace", index=False)

ats_df.to_sql("ats_keywords", conn, if_exists="replace", index=False)

career_df.to_sql("career_paths", conn, if_exists="replace", index=False)

cert_df.to_sql("certifications", conn, if_exists="replace", index=False)

interview_df.to_sql("interview_questions", conn, if_exists="replace", index=False)

salary_df.to_sql("salary_data", conn, if_exists="replace", index=False)

conn.commit()

print("✅ All datasets imported successfully!")

✅ All datasets imported successfully!


In [28]:
tables = [
    "jobs",
    "skills",
    "courses",
    "ats_keywords",
    "career_paths",
    "certifications",
    "interview_questions",
    "salary_data"
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table}: {count} records")

jobs: 300 records
skills: 498 records
courses: 348 records
ats_keywords: 220 records
career_paths: 200 records
certifications: 200 records
interview_questions: 500 records
salary_data: 600 records


In [29]:
query = "SELECT * FROM jobs LIMIT 5"

pd.read_sql(query, conn)

,Job_ID,Job_Title,Company,Experience,Location,Salary,Required_Skills,Preferred_Skills,Education,Employment_Type,Job_Description
0,JOB0001,Data Scientist,TCS,Fresher,Bengaluru,3-5 LPA,Python;Pandas;NumPy;Machine Learning;SQL;Sciki...,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Full Time,TCS is hiring a Data Scientist with strong kno...
1,JOB0002,Data Analyst,Infosys,0-2 Years,Hyderabad,5-8 LPA,SQL;Excel;Power BI;Python;Statistics,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Hybrid,Infosys is hiring a Data Analyst with strong k...
2,JOB0003,ML Engineer,Wipro,2-4 Years,Chennai,8-12 LPA,Python;TensorFlow;PyTorch;ML;Docker,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Remote,Wipro is hiring a ML Engineer with strong know...
3,JOB0004,AI Engineer,Accenture,4-6 Years,Pune,12-18 LPA,Python;LLMs;NLP;TensorFlow;PyTorch,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Full Time,Accenture is hiring a AI Engineer with strong ...
4,JOB0005,Business Analyst,IBM,6+ Years,Mumbai,18-30 LPA,Excel;SQL;Power BI;Communication,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Hybrid,IBM is hiring a Business Analyst with strong k...


In [30]:
pd.read_sql("SELECT * FROM courses LIMIT 5", conn)

,Course_ID,Course_Name,Platform,Skill,Difficulty,Duration,Certificate,Career_Path,Course_URL
0,CRS0001,Python Masterclass 1,Coursera,Python,Beginner,6 Weeks,Yes,Data Analyst,https://learn.example.com/python-1
1,CRS0002,Python Masterclass 2,Udemy,Python,Beginner,8 Weeks,Yes,Data Analyst,https://learn.example.com/python-2
2,CRS0003,Python Masterclass 3,DataCamp,Python,Beginner,12 Weeks,Yes,Data Analyst,https://learn.example.com/python-3
3,CRS0004,Python Masterclass 4,edX,Python,Beginner,4 Weeks,Yes,Data Analyst,https://learn.example.com/python-4
4,CRS0005,Python Masterclass 1,Google,Python,Intermediate,6 Weeks,Yes,Data Analyst,https://learn.example.com/python-1


In [31]:
pd.read_sql("SELECT * FROM skills LIMIT 5", conn)

,Skill_ID,Skill_Name,Category,Difficulty_Level,Demand_Level,Technology_Type,Industry,Importance_Score
0,SK0001,Python,Programming Languages,Beginner,Medium,Technical,"IT, AI & Data Science",61
1,SK0002,Python,Programming Languages,Intermediate,Medium,Technical,"IT, AI & Data Science",62
2,SK0003,Python,Programming Languages,Advanced,Medium,Technical,"IT, AI & Data Science",63
3,SK0004,Python,Programming Languages,Expert,High,Technical,"IT, AI & Data Science",64
4,SK0005,Python,Programming Languages,Industry Standard,High,Technical,"IT, AI & Data Science",65


In [32]:
pd.read_sql("SELECT * FROM career_paths LIMIT 5", conn)

,Path_ID,Current_Role,Next_Role,Required_Skills,Recommended_Courses,Estimated_Time,Difficulty,Average_Salary,Industry
0,PATH0001,Data Analyst,Data Scientist,Python;Machine Learning;Statistics;Scikit-learn,CRS0001;CRS0045;CRS0102,6 Months,Intermediate,₹8-15 LPA,"AI, Data Science & Software"
1,PATH0002,Data Scientist,Senior Data Scientist,Deep Learning;MLOps;AWS;LLMs,CRS0110;CRS0155;CRS0201,12 Months,Advanced,₹15-28 LPA,"AI, Data Science & Software"
2,PATH0003,Data Analyst,BI Developer,Power BI;DAX;SQL;Excel,CRS0030;CRS0060,4 Months,Beginner,₹6-12 LPA,"AI, Data Science & Software"
3,PATH0004,BI Developer,Analytics Manager,Leadership;SQL;Power BI;Python,CRS0090;CRS0120,8 Months,Intermediate,₹12-20 LPA,"AI, Data Science & Software"
4,PATH0005,Python Developer,Backend Developer,FastAPI;Docker;PostgreSQL;Git,CRS0130;CRS0180,5 Months,Intermediate,₹7-14 LPA,"AI, Data Science & Software"


In [33]:
!pip install pypdf python-docx spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [34]:
import os
import re
import sqlite3

import pandas as pd
import numpy as np

from google.colab import files

from pypdf import PdfReader
from docx import Document

import spacy

nlp = spacy.load("en_core_web_sm")

print("✅ Libraries Loaded Successfully")

✅ Libraries Loaded Successfully


In [35]:
uploaded = files.upload()

Saving Bhavesh_Wadhwani_Resume.pdf to Bhavesh_Wadhwani_Resume.pdf


In [36]:
def extract_pdf_text(pdf_path):

    text = ""

    reader = PdfReader(pdf_path)

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [37]:
def extract_docx_text(docx_path):

    document = Document(docx_path)

    text = ""

    for para in document.paragraphs:
        text += para.text + "\n"

    return text

In [38]:
uploaded_file = list(uploaded.keys())[0]

extension = os.path.splitext(uploaded_file)[1].lower()

if extension == ".pdf":

    resume_text = extract_pdf_text(uploaded_file)

elif extension == ".docx":

    resume_text = extract_docx_text(uploaded_file)

else:

    raise Exception("Unsupported File Format")

In [39]:
print("="*100)

print(resume_text[:5000])

print("="*100)

Bhavesh Wadhwani
DATASCIENTIST · GOOGLECLOUDCERTIFIED · MICROSOFTCERTIFIED
 (+91) 899-911-6241 |  bhaveshwadhwani14@gmail.com |  bhaveshwadhwani.github.io |  bhaveshwadhwani |  bhaveshwadhwani
Summary
Results-orientedDataScientist with4.5+years ofwork experienceintheITindustry. Experiencedin Machine-learning,DeepLearning,Com-
puter Vision, Natural Language Processing (NLP), processing real-time data. Recently have verified myData Science/Engineeringskills too by
passing the MicrosoftAzure Data Scientist/GCP Professional Data EngineerExam. Having worked on GCP, Azure, AWS Cloud platforms for
variousdevelopmentanddeploymentprojectsenablesmetohaveabroadviewofthecurrentphaseoftechnologyintheindustry.
WorkExperience
NeosoftTechnologies Pune,India
SENIORCONSULTANT-DATASCIENCE Dec. 2020-Present
• ManaginginternationalClientsas DataScienceConsultant forE&Y Oneof BIG4Consulting firms.
• Helpingtopamongstfortune100clientstomakebetterbusinessdecisionsbasedoninsightsfromtheirdatain supplychai

In [40]:
print("Characters :", len(resume_text))
print("Words      :", len(resume_text.split()))
print("Lines      :", len(resume_text.split("\n")))

Characters : 5182
Words      : 304
Lines      : 74


In [41]:
def clean_resume(text):

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove Email IDs
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove Phone Numbers
    text = re.sub(r"\+?\d[\d\s()-]{8,}\d", " ", text)

    # Remove Extra Spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [42]:
clean_text = clean_resume(resume_text)

print(clean_text[:3000])

Bhavesh Wadhwani DATASCIENTIST · GOOGLECLOUDCERTIFIED · MICROSOFTCERTIFIED  ( |  |  bhaveshwadhwani.github.io |  bhaveshwadhwani |  bhaveshwadhwani Summary Results-orientedDataScientist with4.5+years ofwork experienceintheITindustry. Experiencedin Machine-learning,DeepLearning,Com- puter Vision, Natural Language Processing (NLP), processing real-time data. Recently have verified myData Science/Engineeringskills too by passing the MicrosoftAzure Data Scientist/GCP Professional Data EngineerExam. Having worked on GCP, Azure, AWS Cloud platforms for variousdevelopmentanddeploymentprojectsenablesmetohaveabroadviewofthecurrentphaseoftechnologyintheindustry. WorkExperience NeosoftTechnologies Pune,India SENIORCONSULTANT-DATASCIENCE Dec. 2020-Present • ManaginginternationalClientsas DataScienceConsultant forE&Y Oneof BIG4Consulting firms. • Helpingtopamongstfortune100clientstomakebetterbusinessdecisionsbasedoninsightsfromtheirdatain supplychaindomain . • Optimizeprocurementprocessand dem

In [43]:
doc = nlp(clean_text)

print("Total Tokens :", len(doc))

Total Tokens : 588


In [44]:
skills_df = pd.read_sql("SELECT * FROM skills", conn)

print("Total Skills:", len(skills_df))

skills_df.head()

Total Skills: 498


,Skill_ID,Skill_Name,Category,Difficulty_Level,Demand_Level,Technology_Type,Industry,Importance_Score
0,SK0001,Python,Programming Languages,Beginner,Medium,Technical,"IT, AI & Data Science",61
1,SK0002,Python,Programming Languages,Intermediate,Medium,Technical,"IT, AI & Data Science",62
2,SK0003,Python,Programming Languages,Advanced,Medium,Technical,"IT, AI & Data Science",63
3,SK0004,Python,Programming Languages,Expert,High,Technical,"IT, AI & Data Science",64
4,SK0005,Python,Programming Languages,Industry Standard,High,Technical,"IT, AI & Data Science",65


In [45]:
skill_list = skills_df["Skill_Name"].str.lower().unique().tolist()

print("Total Unique Skills:", len(skill_list))

print(skill_list[:20])

Total Unique Skills: 83
['python', 'java', 'c', 'c++', 'c#', 'r', 'javascript', 'typescript', 'go', 'php', 'kotlin', 'swift', 'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'pytorch', 'xgboost', 'lightgbm', 'eda']


In [47]:
def extract_skills(text, skills):

    text = text.lower()

    found_skills = []

    for skill in skills:
        if skill in text:
            found_skills.append(skill)

    return sorted(list(set(found_skills)))

In [48]:
resume_skills = extract_skills(clean_text, skill_list)

print("Skills Found:", len(resume_skills))

Skills Found: 29


In [49]:
for skill in resume_skills:
    print("✓", skill)

✓ aws
✓ azure
✓ c
✓ css
✓ docker
✓ eda
✓ flask
✓ gcp
✓ git
✓ github
✓ go
✓ html
✓ java
✓ javascript
✓ matplotlib
✓ mongodb
✓ mysql
✓ nltk
✓ numpy
✓ pandas
✓ php
✓ plotly
✓ python
✓ pytorch
✓ r
✓ seaborn
✓ sql
✓ sqlite
✓ tensorflow


In [50]:
resume_skill_df = pd.DataFrame({
    "Detected Skills": resume_skills
})

resume_skill_df

,Detected Skills
0,aws
1,azure
2,c
3,css
4,docker
5,eda
6,flask
7,gcp
8,git
9,github


In [51]:
total_skills = len(skill_list)

matched_skills = len(resume_skills)

match_percentage = (matched_skills / total_skills) * 100

print(f"Matched Skills : {matched_skills}")
print(f"Database Skills: {total_skills}")
print(f"Skill Match : {match_percentage:.2f}%")

Matched Skills : 29
Database Skills: 83
Skill Match : 34.94%


In [52]:
matched_df = skills_df[
    skills_df["Skill_Name"].str.lower().isin(resume_skills)
]

matched_df.sort_values(
    by="Importance_Score",
    ascending=False
).head(20)

,Skill_ID,Skill_Name,Category,Difficulty_Level,Demand_Level,Technology_Type,Industry,Importance_Score
39,SK0040,JavaScript,Programming Languages,Expert,High,Technical,"IT, AI & Data Science",100
80,SK0081,NumPy,Data Science,Advanced,Medium,Technical,"IT, AI & Data Science",100
285,SK0286,Docker,Cloud & DevOps,Expert,High,Technical,"IT, AI & Data Science",100
367,SK0368,HTML,Web Development,Intermediate,Medium,Technical,"IT, AI & Data Science",100
244,SK0245,SQLite,Databases,Industry Standard,High,Technical,"IT, AI & Data Science",100
38,SK0039,JavaScript,Programming Languages,Advanced,Medium,Technical,"IT, AI & Data Science",99
79,SK0080,NumPy,Data Science,Intermediate,Medium,Technical,"IT, AI & Data Science",99
243,SK0244,SQLite,Databases,Expert,High,Technical,"IT, AI & Data Science",99
366,SK0367,HTML,Web Development,Beginner,Medium,Technical,"IT, AI & Data Science",99
284,SK0285,Docker,Cloud & DevOps,Advanced,Medium,Technical,"IT, AI & Data Science",99


In [53]:
print("clean_text exists:", "clean_text" in globals())
print("skill_list exists:", "skill_list" in globals())
print("extract_skills exists:", "extract_skills" in globals())

clean_text exists: True
skill_list exists: True
extract_skills exists: True


In [54]:
import re

def extract_skills(text, skill_list):

    text = text.lower()

    detected_skills = []

    for skill in skill_list:

        pattern = r'\b' + re.escape(skill.lower()) + r'\b'

        if re.search(pattern, text):
            detected_skills.append(skill)

    return sorted(list(set(detected_skills)))

In [55]:
resume_skills = extract_skills(clean_text, skill_list)

print(f"Total Skills Detected: {len(resume_skills)}")

Total Skills Detected: 23


In [56]:
for i, skill in enumerate(resume_skills, start=1):
    print(f"{i}. {skill}")

1. aws
2. azure
3. css
4. docker
5. flask
6. gcp
7. git
8. github
9. html
10. javascript
11. matplotlib
12. mongodb
13. mysql
14. nltk
15. numpy
16. pandas
17. plotly
18. python
19. pytorch
20. seaborn
21. sql
22. sqlite
23. tensorflow


In [57]:
resume_skill_df = pd.DataFrame({
    "Detected_Skills": resume_skills
})

resume_skill_df

,Detected_Skills
0,aws
1,azure
2,css
3,docker
4,flask
5,gcp
6,git
7,github
8,html
9,javascript


In [58]:
matched_skills_df = skills_df[
    skills_df["Skill_Name"].str.lower().isin(
        [s.lower() for s in resume_skills]
    )
]

matched_skills_df

,Skill_ID,Skill_Name,Category,Difficulty_Level,Demand_Level,Technology_Type,Industry,Importance_Score
0,SK0001,Python,Programming Languages,Beginner,Medium,Technical,"IT, AI & Data Science",61
1,SK0002,Python,Programming Languages,Intermediate,Medium,Technical,"IT, AI & Data Science",62
2,SK0003,Python,Programming Languages,Advanced,Medium,Technical,"IT, AI & Data Science",63
3,SK0004,Python,Programming Languages,Expert,High,Technical,"IT, AI & Data Science",64
4,SK0005,Python,Programming Languages,Industry Standard,High,Technical,"IT, AI & Data Science",65
...,...,...,...,...,...,...,...,...
391,SK0392,Flask,Web Development,Intermediate,Medium,Technical,"IT, AI & Data Science",83
392,SK0393,Flask,Web Development,Advanced,Medium,Technical,"IT, AI & Data Science",84
393,SK0394,Flask,Web Development,Expert,High,Technical,"IT, AI & Data Science",85
394,SK0395,Flask,Web Development,Industry Standard,High,Technical,"IT, AI & Data Science",86


In [59]:
matched_skills_df["Category"].value_counts()

,count
Category,
Cloud & DevOps,36
Databases,24
Data Science,24
Visualization,18
Web Development,18
Programming Languages,12
NLP & AI,6


In [60]:
total_importance = matched_skills_df["Importance_Score"].sum()

print("Total Skill Importance Score:", total_importance)

Total Skill Importance Score: 11241


In [61]:
ats_df = pd.read_sql("SELECT * FROM ats_keywords", conn)

print("Total ATS Keywords:", len(ats_df))

ats_df.head()

Total ATS Keywords: 220


,Keyword_ID,Keyword,Category,Importance,Weight,Mandatory,Synonym,Industry
0,ATS0001,Python,Programming,Mandatory,10,Yes,"Py, Python3","IT, AI & Data Science"
1,ATS0002,Python,Programming,Important,8,No,"Py, Python3","IT, AI & Data Science"
2,ATS0003,Python,Programming,Preferred,6,No,"Py, Python3","IT, AI & Data Science"
3,ATS0004,Python,Programming,Optional,4,No,"Py, Python3","IT, AI & Data Science"
4,ATS0005,Java,Programming,Mandatory,10,Yes,"Core Java, Java SE","IT, AI & Data Science"


In [62]:
ats_keywords = ats_df["Keyword"].str.lower().unique().tolist()

print("Total ATS Keywords:", len(ats_keywords))

Total ATS Keywords: 55


In [63]:
import re

def extract_ats_keywords(text, keyword_list):

    text = text.lower()

    matched_keywords = []

    for keyword in keyword_list:

        pattern = r"\b" + re.escape(keyword.lower()) + r"\b"

        if re.search(pattern, text):
            matched_keywords.append(keyword)

    return sorted(list(set(matched_keywords)))

In [64]:
matched_keywords = extract_ats_keywords(clean_text, ats_keywords)

print("Matched ATS Keywords:", len(matched_keywords))

Matched ATS Keywords: 15


In [65]:
for keyword in matched_keywords:
    print("✓", keyword)

✓ aws
✓ azure
✓ docker
✓ gcp
✓ git
✓ github
✓ javascript
✓ mongodb
✓ mysql
✓ nltk
✓ python
✓ pytorch
✓ sql
✓ sqlite
✓ tensorflow


In [66]:
matched_keywords_df = ats_df[
    ats_df["Keyword"].str.lower().isin(
        [k.lower() for k in matched_keywords]
    )
]

matched_keywords_df

,Keyword_ID,Keyword,Category,Importance,Weight,Mandatory,Synonym,Industry
0,ATS0001,Python,Programming,Mandatory,10,Yes,"Py, Python3","IT, AI & Data Science"
1,ATS0002,Python,Programming,Important,8,No,"Py, Python3","IT, AI & Data Science"
2,ATS0003,Python,Programming,Preferred,6,No,"Py, Python3","IT, AI & Data Science"
3,ATS0004,Python,Programming,Optional,4,No,"Py, Python3","IT, AI & Data Science"
12,ATS0013,JavaScript,Programming,Mandatory,10,Yes,"JS, ECMAScript","IT, AI & Data Science"
13,ATS0014,JavaScript,Programming,Important,8,No,"JS, ECMAScript","IT, AI & Data Science"
14,ATS0015,JavaScript,Programming,Preferred,6,No,"JS, ECMAScript","IT, AI & Data Science"
15,ATS0016,JavaScript,Programming,Optional,4,No,"JS, ECMAScript","IT, AI & Data Science"
16,ATS0017,SQL,Programming,Mandatory,10,Yes,Structured Query Language,"IT, AI & Data Science"
17,ATS0018,SQL,Programming,Important,8,No,Structured Query Language,"IT, AI & Data Science"


In [67]:
keyword_score = (
    len(matched_keywords) / len(ats_keywords)
) * 100

print(f"ATS Keyword Score: {keyword_score:.2f}%")

ATS Keyword Score: 27.27%


In [68]:
word_count = len(clean_text.split())

print("Resume Word Count:", word_count)

if word_count < 300:
    length_score = 10
elif word_count <= 700:
    length_score = 20
else:
    length_score = 15

print("Length Score:", length_score)

Resume Word Count: 302
Length Score: 20


In [69]:
skill_score = min(len(resume_skills) * 2, 40)

print("Skill Score:", skill_score)

Skill Score: 40


In [70]:
overall_ats_score = (
    skill_score +
    length_score +
    (keyword_score * 0.4)
)

overall_ats_score = min(round(overall_ats_score, 2), 100)

print("=" * 40)
print(f"Overall ATS Score: {overall_ats_score}/100")
print("=" * 40)

Overall ATS Score: 70.91/100


In [71]:
if overall_ats_score >= 90:
    rating = "Excellent ⭐⭐⭐⭐⭐"
elif overall_ats_score >= 75:
    rating = "Very Good ⭐⭐⭐⭐"
elif overall_ats_score >= 60:
    rating = "Good ⭐⭐⭐"
elif overall_ats_score >= 45:
    rating = "Average ⭐⭐"
else:
    rating = "Needs Improvement ⭐"

print("Resume Rating:", rating)

Resume Rating: Good ⭐⭐⭐


In [72]:
jobs_df = pd.read_sql("SELECT * FROM jobs", conn)

print("Total Jobs:", len(jobs_df))

jobs_df.head()

Total Jobs: 300


,Job_ID,Job_Title,Company,Experience,Location,Salary,Required_Skills,Preferred_Skills,Education,Employment_Type,Job_Description
0,JOB0001,Data Scientist,TCS,Fresher,Bengaluru,3-5 LPA,Python;Pandas;NumPy;Machine Learning;SQL;Sciki...,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Full Time,TCS is hiring a Data Scientist with strong kno...
1,JOB0002,Data Analyst,Infosys,0-2 Years,Hyderabad,5-8 LPA,SQL;Excel;Power BI;Python;Statistics,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Hybrid,Infosys is hiring a Data Analyst with strong k...
2,JOB0003,ML Engineer,Wipro,2-4 Years,Chennai,8-12 LPA,Python;TensorFlow;PyTorch;ML;Docker,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Remote,Wipro is hiring a ML Engineer with strong know...
3,JOB0004,AI Engineer,Accenture,4-6 Years,Pune,12-18 LPA,Python;LLMs;NLP;TensorFlow;PyTorch,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Full Time,Accenture is hiring a AI Engineer with strong ...
4,JOB0005,Business Analyst,IBM,6+ Years,Mumbai,18-30 LPA,Excel;SQL;Power BI;Communication,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Hybrid,IBM is hiring a Business Analyst with strong k...


In [73]:
def recommend_jobs(resume_skills, jobs_df):

    recommendations = []

    resume_skill_set = set(skill.lower() for skill in resume_skills)

    for _, row in jobs_df.iterrows():

        job_skills = [
            s.strip().lower()
            for s in row["Required_Skills"].split(";")
        ]

        matched = resume_skill_set.intersection(job_skills)

        score = (len(matched) / len(job_skills)) * 100

        recommendations.append({

            "Job Title": row["Job_Title"],

            "Company": row["Company"],

            "Location": row["Location"],

            "Salary": row["Salary"],

            "Match %": round(score, 2),

            "Matched Skills": ", ".join(sorted(matched))

        })

    return pd.DataFrame(recommendations)

In [74]:
job_recommendations = recommend_jobs(
    resume_skills,
    jobs_df
)

job_recommendations.head()

,Job Title,Company,Location,Salary,Match %,Matched Skills
0,Data Scientist,TCS,Bengaluru,3-5 LPA,66.67,"numpy, pandas, python, sql"
1,Data Analyst,Infosys,Hyderabad,5-8 LPA,40.00,"python, sql"
2,ML Engineer,Wipro,Chennai,8-12 LPA,80.00,"docker, python, pytorch, tensorflow"
3,AI Engineer,Accenture,Pune,12-18 LPA,60.00,"python, pytorch, tensorflow"
4,Business Analyst,IBM,Mumbai,18-30 LPA,25.00,sql


In [75]:
top_jobs = job_recommendations.sort_values(
    by="Match %",
    ascending=False
).head(5)

top_jobs

,Job Title,Company,Location,Salary,Match %,Matched Skills
2,ML Engineer,Wipro,Chennai,8-12 LPA,80.0,"docker, python, pytorch, tensorflow"
242,ML Engineer,Wipro,Remote,8-12 LPA,80.0,"docker, python, pytorch, tensorflow"
42,ML Engineer,Oracle,Noida,8-12 LPA,80.0,"docker, python, pytorch, tensorflow"
62,ML Engineer,Wipro,Remote,8-12 LPA,80.0,"docker, python, pytorch, tensorflow"
162,ML Engineer,Oracle,Bengaluru,8-12 LPA,80.0,"docker, python, pytorch, tensorflow"


In [76]:
import re

def contact_score(text):

    score = 0

    # Email
    if re.search(r"\S+@\S+\.\S+", text):
        score += 2

    # Phone Number
    if re.search(r"(\+?\d[\d\s()-]{8,}\d)", text):
        score += 2

    # LinkedIn or GitHub
    if "linkedin" in text.lower() or "github" in text.lower():
        score += 1

    return score

In [77]:
contact_marks = contact_score(resume_text)

print("Contact Score:", contact_marks, "/5")

Contact Score: 5 /5


In [78]:
def section_score(text):

    score = 0

    sections = [

        "summary",

        "education",

        "experience",

        "skills",

        "projects",

        "certification",

        "internship"

    ]

    lower_text = text.lower()

    for section in sections:

        if section in lower_text:

            score += 1

    return min(score,10)

In [79]:
section_marks = section_score(clean_text)

print("Section Score:", section_marks)

Section Score: 6


In [80]:
keyword_marks = min(
    len(matched_keywords),
    20
)

print(keyword_marks)

15


In [81]:
def education_score(text):

    education_keywords = [

        "b.e",

        "b.tech",

        "m.tech",

        "mca",

        "bca",

        "b.sc",

        "m.sc",

        "phd",

        "bachelor",

        "master"

    ]

    score = 0

    lower = text.lower()

    for word in education_keywords:

        if word in lower:

            score += 2

    return min(score,10)

In [82]:
education_marks = education_score(clean_text)

print(education_marks)

2


In [83]:
def experience_score(text):

    match = re.search(r'(\d+)\+?\s*years?', text.lower())

    if match:

        years = int(match.group(1))

        if years >= 5:

            return 10

        elif years >= 3:

            return 8

        elif years >= 1:

            return 6

    return 2

In [84]:
experience_marks = experience_score(clean_text)

print(experience_marks)

10


In [85]:
def certification_score(text):

    words = [

        "certified",

        "certificate",

        "certification",

        "google",

        "microsoft",

        "aws",

        "ibm"

    ]

    score = 0

    lower = text.lower()

    for word in words:

        if word in lower:

            score += 1

    return min(score,5)

In [86]:
def project_score(text):

    keywords = [

        "project",

        "projects",

        "github",

        "portfolio",

        "kaggle"

    ]

    score = 0

    lower = text.lower()

    for word in keywords:

        if word in lower:

            score += 1

    return min(score,5)

In [87]:
words = len(clean_text.split())

if 350 <= words <= 800:

    length_marks = 5

elif 250 <= words <= 1000:

    length_marks = 4

else:

    length_marks = 2

print(length_marks)

4


In [89]:
# Calculate skill score (Maximum 30)

skill_marks = min(len(resume_skills), 30)

print("Skill Score:", skill_marks, "/30")

Skill Score: 23 /30


In [90]:
overall_score = (

    contact_marks +

    section_marks +

    skill_marks +

    keyword_marks +

    education_marks +

    experience_marks +

    certification_score(clean_text) +

    project_score(clean_text) +

    length_marks

)

print("="*40)

print(f"Professional ATS Score : {overall_score}/100")

print("="*40)

Professional ATS Score : 73/100


In [92]:
def calculate_ats_score(clean_text, resume_skills):

    # Contact
    contact_marks = contact_score(clean_text)

    # Sections
    section_marks = section_score(clean_text)

    # Skills
    skill_marks = min(len(resume_skills), 30)

    # ATS Keywords
    matched_keywords = extract_ats_keywords(clean_text, ats_keywords)
    keyword_marks = min(len(matched_keywords), 20)

    # Education
    education_marks = education_score(clean_text)

    # Experience
    experience_marks = experience_score(clean_text)

    # Certifications
    certification_marks = certification_score(clean_text)

    # Projects
    project_marks = project_score(clean_text)

    # Resume Length
    words = len(clean_text.split())

    if 350 <= words <= 800:
        length_marks = 5
    elif 250 <= words <= 1000:
        length_marks = 4
    else:
        length_marks = 2

    overall_score = (
        contact_marks +
        section_marks +
        skill_marks +
        keyword_marks +
        education_marks +
        experience_marks +
        certification_marks +
        project_marks +
        length_marks
    )

    return {
        "Overall Score": overall_score,
        "Contact": contact_marks,
        "Sections": section_marks,
        "Skills": skill_marks,
        "Keywords": keyword_marks,
        "Education": education_marks,
        "Experience": experience_marks,
        "Certifications": certification_marks,
        "Projects": project_marks,
        "Length": length_marks,
    }

In [93]:
ats_result = calculate_ats_score(clean_text, resume_skills)

In [94]:
for key, value in ats_result.items():
    print(f"{key:<18}: {value}")

Overall Score     : 69
Contact           : 1
Sections          : 6
Skills            : 23
Keywords          : 15
Education         : 2
Experience        : 10
Certifications    : 5
Projects          : 3
Length            : 4


In [95]:
############################################################
# MODULE 1
# PROFESSIONAL ATS ENGINE
############################################################

def calculate_ats_score(
    resume_text,
    clean_text,
    resume_skills,
    ats_keywords
):
    """
    Calculate a professional ATS score and return
    both the total score and the detailed breakdown.
    """

    # Contact
    contact = contact_score(resume_text)

    # Sections
    sections = section_score(clean_text)

    # Skills
    skills = min(len(resume_skills), 30)

    # ATS Keywords
    matched_keywords = extract_ats_keywords(
        clean_text,
        ats_keywords
    )

    keywords = min(len(matched_keywords), 20)

    # Education
    education = education_score(clean_text)

    # Experience
    experience = experience_score(clean_text)

    # Certifications
    certifications = certification_score(clean_text)

    # Projects
    projects = project_score(clean_text)

    # Resume Length
    words = len(clean_text.split())

    if 350 <= words <= 800:
        length = 5
    elif 250 <= words <= 1000:
        length = 4
    else:
        length = 2

    total = (
        contact
        + sections
        + skills
        + keywords
        + education
        + experience
        + certifications
        + projects
        + length
    )

    total = min(total, 100)

    return {
        "Overall Score": total,
        "Contact": contact,
        "Sections": sections,
        "Skills": skills,
        "Keywords": keywords,
        "Education": education,
        "Experience": experience,
        "Certifications": certifications,
        "Projects": projects,
        "Length": length,
        "Matched Keywords": matched_keywords
    }

In [96]:
ats_result = calculate_ats_score(
    resume_text,
    clean_text,
    resume_skills,
    ats_keywords
)

In [97]:
print("=" * 60)
print("PROFESSIONAL ATS REPORT")
print("=" * 60)

for key, value in ats_result.items():

    if key != "Matched Keywords":
        print(f"{key:<20}: {value}")

PROFESSIONAL ATS REPORT
Overall Score       : 73
Contact             : 5
Sections            : 6
Skills              : 23
Keywords            : 15
Education           : 2
Experience          : 10
Certifications      : 5
Projects            : 3
Length              : 4


In [98]:
print("\nMatched ATS Keywords\n")

for keyword in ats_result["Matched Keywords"]:
    print("✓", keyword)


Matched ATS Keywords

✓ aws
✓ azure
✓ docker
✓ gcp
✓ git
✓ github
✓ javascript
✓ mongodb
✓ mysql
✓ nltk
✓ python
✓ pytorch
✓ sql
✓ sqlite
✓ tensorflow


In [99]:
############################################################
# MODULE 2
# JOB RECOMMENDATION ENGINE
############################################################

jobs_df = pd.read_sql("SELECT * FROM jobs", conn)

print(f"Total Jobs Available : {len(jobs_df)}")

jobs_df.head()

Total Jobs Available : 300


,Job_ID,Job_Title,Company,Experience,Location,Salary,Required_Skills,Preferred_Skills,Education,Employment_Type,Job_Description
0,JOB0001,Data Scientist,TCS,Fresher,Bengaluru,3-5 LPA,Python;Pandas;NumPy;Machine Learning;SQL;Sciki...,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Full Time,TCS is hiring a Data Scientist with strong kno...
1,JOB0002,Data Analyst,Infosys,0-2 Years,Hyderabad,5-8 LPA,SQL;Excel;Power BI;Python;Statistics,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Hybrid,Infosys is hiring a Data Analyst with strong k...
2,JOB0003,ML Engineer,Wipro,2-4 Years,Chennai,8-12 LPA,Python;TensorFlow;PyTorch;ML;Docker,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Remote,Wipro is hiring a ML Engineer with strong know...
3,JOB0004,AI Engineer,Accenture,4-6 Years,Pune,12-18 LPA,Python;LLMs;NLP;TensorFlow;PyTorch,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Full Time,Accenture is hiring a AI Engineer with strong ...
4,JOB0005,Business Analyst,IBM,6+ Years,Mumbai,18-30 LPA,Excel;SQL;Power BI;Communication,Communication;Problem Solving;Git;Teamwork,B.E/B.Tech/M.Tech/MCA,Hybrid,IBM is hiring a Business Analyst with strong k...


In [100]:
def recommend_jobs(resume_skills, jobs_df):

    recommendations = []

    resume_skill_set = set(skill.lower() for skill in resume_skills)

    for _, row in jobs_df.iterrows():

        required_skills = [
            skill.strip().lower()
            for skill in str(row["Required_Skills"]).split(";")
            if skill.strip()
        ]

        preferred_skills = [
            skill.strip().lower()
            for skill in str(row["Preferred_Skills"]).split(";")
            if skill.strip()
        ]

        matched_required = sorted(
            list(resume_skill_set.intersection(required_skills))
        )

        matched_preferred = sorted(
            list(resume_skill_set.intersection(preferred_skills))
        )

        missing_skills = sorted(
            list(set(required_skills) - resume_skill_set)
        )

        total_required = len(required_skills)

        if total_required == 0:
            match_percentage = 0
        else:
            match_percentage = round(
                (len(matched_required) / total_required) * 100,
                2
            )

        recommendations.append({

            "Job Title": row["Job_Title"],

            "Company": row["Company"],

            "Location": row["Location"],

            "Salary": row["Salary"],

            "Match %": match_percentage,

            "Matched Skills": matched_required,

            "Preferred Skills": matched_preferred,

            "Missing Skills": missing_skills,

            "Experience": row["Experience"],

            "Education": row["Education"]

        })

    return pd.DataFrame(recommendations)

In [101]:
job_results = recommend_jobs(
    resume_skills,
    jobs_df
)

print("Recommendations Generated Successfully")

Recommendations Generated Successfully


In [102]:
top_jobs = (
    job_results
    .sort_values("Match %", ascending=False)
    .reset_index(drop=True)
)

top_jobs.head(10)

,Job Title,Company,Location,Salary,Match %,Matched Skills,Preferred Skills,Missing Skills,Experience,Education
0,ML Engineer,Wipro,Chennai,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
1,ML Engineer,Wipro,Remote,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
2,ML Engineer,Oracle,Noida,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
3,ML Engineer,Wipro,Remote,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
4,ML Engineer,Oracle,Bengaluru,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
5,ML Engineer,Deloitte,Kolkata,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
6,ML Engineer,Wipro,Chennai,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
7,ML Engineer,Oracle,Pune,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
8,ML Engineer,Wipro,Delhi,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA
9,ML Engineer,Deloitte,Mumbai,8-12 LPA,80.0,"[docker, python, pytorch, tensorflow]",[git],[ml],2-4 Years,B.E/B.Tech/M.Tech/MCA


In [103]:
best_job = top_jobs.iloc[0]

print("=" * 60)

print("BEST JOB MATCH")

print("=" * 60)

print("Job Title      :", best_job["Job Title"])
print("Company        :", best_job["Company"])
print("Location       :", best_job["Location"])
print("Salary         :", best_job["Salary"])
print("Match %        :", best_job["Match %"])

print("\nMatched Skills")
print(best_job["Matched Skills"])

print("\nMissing Skills")
print(best_job["Missing Skills"])

BEST JOB MATCH
Job Title      : ML Engineer
Company        : Wipro
Location       : Chennai
Salary         : 8-12 LPA
Match %        : 80.0

Matched Skills
['docker', 'python', 'pytorch', 'tensorflow']

Missing Skills
['ml']


In [104]:
top_jobs[
    [
        "Job Title",
        "Company",
        "Location",
        "Salary",
        "Match %"
    ]
].head(5)

,Job Title,Company,Location,Salary,Match %
0,ML Engineer,Wipro,Chennai,8-12 LPA,80.0
1,ML Engineer,Wipro,Remote,8-12 LPA,80.0
2,ML Engineer,Oracle,Noida,8-12 LPA,80.0
3,ML Engineer,Wipro,Remote,8-12 LPA,80.0
4,ML Engineer,Oracle,Bengaluru,8-12 LPA,80.0
